In [ ]:
import os
import numpy as np
import pandas as pd
import commot as ct
import scanpy as sc
import matplotlib.pyplot as plt

In [ ]:
df_cellchat = ct.pp.ligand_receptor_database(species='human', signaling_type='Secreted Signaling', database='CellChat')
df_cellchat

In [ ]:
df_cellchat_filtered = ct.pp.filter_lr_database(df_cellchat, adata, min_cell_pct=0.001)

In [ ]:
import os 
sp_path = '../data/h5ad_fragment'
file_list = os.listdir(sp_path)
file_list.sort()

In [ ]:
from scipy.spatial import cKDTree

adata_list = []
cellchat_list_common = []

for idx, file in enumerate(file_list):
    try:
        adata = sc.read_h5ad(f'{sp_path}/{file}')
        df_cellchat_filtered = ct.pp.filter_lr_database(df_cellchat, adata, min_cell_pct=0.001)
        coords = adata.obsm["spatial"]

        edge_malignant_cell = adata.obs['area_Cancer_Epithelial_filtered'] == 'edge'
        edge_malignant_coords = coords[edge_malignant_cell.values]

        # Build a KDTree for fast spatial distance calculation
        tree = cKDTree(edge_malignant_coords)
        distances, _ = tree.query(coords)

        # Add the distances to adata.obs
        adata.obs["Distance_from_Cancer_boundary"] = distances
        adata.obs['peritumoral_area'] = np.where(adata.obs["Distance_from_Cancer_boundary"] < 75, 1, 0)
        adata_sub = adata[adata.obs['peritumoral_area'] == 1].copy()
        
        ct.tl.spatial_communication(adata_sub, database_name='cellchat', 
                                    df_ligrec=df_cellchat_filtered, dis_thr=50, 
                                    heteromeric=True, pathway_sum=True)
        adata_list.append(adata_sub)
        adata_sub.write_h5ad(f'../data/commot_result_sub/{file}')
        cellchat_list = np.array([i.split('commot-cellchat-')[-1] 
                                  for i in adata_sub.obsp.keys() if 'commot-cellchat' in i])
        if idx == 0:
            cellchat_list_common = cellchat_list
        else:
            cellchat_list_common = np.intersect1d(cellchat_list, cellchat_list_common)
    except MemoryError:
        print(f"MemoryError: skipping file '{file}'.")
        continue
    except Exception as e:
        print(f"Error in file '{file}': {e}. Skipping.")
        continue

print(cellchat_list_common)


In [ ]:
ordered_result_all = []
batch_list = []

for adata in adata_list:
    # Extract batch name
    batch = adata.obs["batch"].unique()[0]
    batch_list.append(batch)

    # Count non-zero interactions for each pathway
    pathway_counts = []

    for pathway in adata.obsp.keys():
        if pathway == "commot-cellchat-total-total":
            continue

        interaction_count = adata.obsp[pathway].nnz

        if interaction_count > 0:
            pathway_name = pathway.removeprefix("commot-cellchat-")
            pathway_counts.append((pathway_name, interaction_count))

    # Sort pathways by interaction count
    pathway_counts.sort(key=lambda x: x[1], reverse=True)

    ordered_result = [pathway for pathway, count in pathway_counts]
    ordered_result_all.append(ordered_result)

# Convert ranked pathway lists to DataFrame
commot_ranked_result = pd.DataFrame(
    ordered_result_all,
    index=batch_list
)

commot_ranked_result.to_csv(
    "../data/commot_ranked_result.csv"
)

In [ ]:
result_total = pd.read_csv('../data/results_df_spatial_vessel_linkerenz.csv', index_col = 0)

In [ ]:
adata_list_pos = []
adata_list_low = []

for adata in adata_list:
    batch = adata.obs['batch'].unique()[0]

    # Get the HER2_type value (Series -> scalar)
    ct_series = result_total.loc[result_total['batch'] == batch, 'HER2_type']

    if ct_series.empty:
        # If there is no matching batch, skip it or handle the error
        print(f"[WARN] batch '{batch}' is not present in result_total.")
        continue

    clinical_type = ct_series.iloc[0]        # or .values[0] / .squeeze()

    if clinical_type == 'Pos':
        adata_list_pos.append(adata)
    elif clinical_type == 'Low':
        adata_list_low.append(adata)


In [ ]:
results_pos = {}
for interaction in LR_positive:
    try:
        result = analyze_interaction(
            interaction,
            adata_list_pos,
            df_cellchat,
            out_dir="../figures/LR_plots_pos",
            plot_kind="both"
        )
        results_pos[interaction] = result
    except ValueError as e:
        # Print a warning only and move on to the next interaction
        print(f"[SKIP] {interaction}: {e}")


In [ ]:
results_low = {}
for interaction in LR_low:
    try:
        result = analyze_interaction(
            interaction,
            adata_list_low,
            df_cellchat,
            out_dir="../figures/LR_plots_low",
            plot_kind="both"
        )
        results_low[interaction] = result
    except ValueError as e:
        # Print a warning only and move on to the next interaction
        print(f"[SKIP] {interaction}: {e}")
